In [7]:
from pathlib import Path
import sys

# Point directly to the folder containing your dynacir directory
qiskit_workspace = Path(r"C:\Users\Raghav\OneDrive\Desktop\QML\Qiskit")

# Append it to sys.path if it isn't already there
if str(qiskit_workspace) not in sys.path:
    sys.path.insert(0, str(qiskit_workspace))

print(f"✅ Python path updated to scan: {qiskit_workspace}")

✅ Python path updated to scan: C:\Users\Raghav\OneDrive\Desktop\QML\Qiskit


In [9]:
from pathlib import Path

# Target the true path of your module
dynacir_path = Path(r"C:\Users\Raghav\OneDrive\Desktop\QML\Qiskit\dynacir")
passes_path = dynacir_path / "passes"

# Ensure the subfolder exists and create the required __init__.py files
passes_path.mkdir(parents=True, exist_ok=True)
(dynacir_path / "__init__.py").touch(exist_ok=True)
(passes_path / "__init__.py").touch(exist_ok=True)

print("📁 Package initialization files (__init__.py) verified and created!")

📁 Package initialization files (__init__.py) verified and created!


In [12]:
import os
from pathlib import Path

passes_dir = Path(r"C:\Users\Raghav\OneDrive\Desktop\QML\Qiskit\dynacir\passes")
print("Files in passes directory:", [f.name for f in passes_dir.glob("*.py")])

Files in passes directory: ['__init__.py']


In [13]:
from pathlib import Path

passes_dir = Path(r"C:\Users\Raghav\OneDrive\Desktop\QML\Qiskit\dynacir\passes")

# 1. Define the complete implementation for CollectResets
collect_resets_code = """from qiskit.transpiler.basepasses import AnalysisPass

class CollectResets(AnalysisPass):
    \"\"\"A custom analysis pass to collect and track mid-circuit reset/measurement operations.\"\"\"
    def __init__(self):
        super().__init__()
        self.resets = []

    def run(self, dag):
        self.resets.clear()
        for node in dag.op_nodes():
            if node.name == 'reset' or (node.name == 'measure' and getattr(node.op, 'condition', None) is not None):
                self.resets.append(node)
        self.property_set['collected_resets'] = self.resets
        return dag
"""

# 2. Write the file to disk
(passes_dir / "collect_resets.py").write_text(collect_resets_code)

# 3. Expose it via __init__.py for clean importing
(passes_dir / "__init__.py").write_text("from .collect_resets import CollectResets\n")

print("✨ CollectResets class successfully written and registered!")

✨ CollectResets class successfully written and registered!


In [23]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.circuit.controlflow import IfElseOp, WhileLoopOp

# ==========================================
# 1. Initialize and Configure Target Backend
# ==========================================
backend = GenericBackendV2(num_qubits=12)

# Explicitly register dynamic control flow operations to the target
backend.target.add_instruction(IfElseOp, name="if_else")
backend.target.add_instruction(WhileLoopOp, name="while_loop")

print(
    "✅ Target backend verified with operations:", list(backend.target.operation_names)
)


# ==========================================
# 2. Define Sub-Circuits for the Branches
# ==========================================
# For a structural if_else, we build explicit sub-circuits for the True and False paths.
# They must match the qubit and classical bit allocation of the targeting block.
true_body = QuantumCircuit(1)
true_body.x(0)  # Apply correction if condition met

false_body = QuantumCircuit(1)
false_body.id(0)  # Do nothing/identity if condition not met


# ==========================================
# 3. Generate TFIM Trotter Step Circuit
# ==========================================
def generate_tfim_trotter_step(
    num_qubits: int,
    J: float,
    g: float,
    dt: float,
    true_block: QuantumCircuit,
    false_block: QuantumCircuit,
) -> QuantumCircuit:
    """
    Generates a single Trotter step for a 1D TFIM chain with functional dynamic control flow.
    """
    qr = QuantumRegister(num_qubits, name="q")
    cr_measure = ClassicalRegister(num_qubits, name="mid_meas")
    qc = QuantumCircuit(qr, cr_measure)

    # --- Step A: Transverse Field Term (X-rotations) ---
    for i in range(num_qubits):
        qc.rx(2 * g * dt, qr[i])

    qc.barrier()

    # --- Step B: Ising Interaction Term (ZZ-rotations via interleaved pairs) ---
    for i in range(0, num_qubits - 1, 2):
        qc.cx(qr[i], qr[i + 1])
        qc.rz(2 * J * dt, qr[i + 1])
        qc.cx(qr[i], qr[i + 1])

    for i in range(1, num_qubits - 1, 2):
        qc.cx(qr[i], qr[i + 1])
        qc.rz(2 * J * dt, qr[i + 1])
        qc.cx(qr[i], qr[i + 1])

    qc.barrier()

    # --- Step C: Dynamic Mid-Circuit Measurement ---
    qc.measure(qr[0], cr_measure[0])

    # --- Step D: Functional Control Flow Application ---
    # Arguments expected by structural if_else:
    # (classical_cond, true_body, false_body, qubits, clbits)
    qc.if_else(
        (cr_measure[0], 1),
        true_block,
        false_block,
        [qr[0]],  # Qubits passed to the sub-blocks
        [],  # Classical bits passed to the sub-blocks (none needed here)
    )

    return qc


# ==========================================
# 4. Instantiate and Inspect Circuit
# ==========================================
num_qubits = 12
J_param = 1.0
g_param = 0.5
time_step = 0.05

tfim_circuit = generate_tfim_trotter_step(
    num_qubits, J_param, g_param, time_step, true_body, false_body
)

print(f"📦 TFIM circuit successfully built!")
print(f"Circuit Depth: {tfim_circuit.depth()}")
print(f"Circuit Operations Count: {dict(tfim_circuit.count_ops())}")

✅ Target backend verified with operations: ['cx', 'id', 'rz', 'sx', 'x', 'reset', 'delay', 'measure', 'if_else', 'while_loop']
📦 TFIM circuit successfully built!
Circuit Depth: 9
Circuit Operations Count: {'cx': 22, 'rx': 12, 'rz': 11, 'barrier': 2, 'measure': 1, 'if_else': 1}
